# Data Integration

In [1]:
import scanpy as sc
import anndata

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)

In [2]:
adata = anndata.read_h5ad("../annotation/annotation.h5ad.gz")
adata

/opt/conda/envs/integration/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/opt/conda/envs/integration/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 892 × 56134
    obs: 'sample', 'fastq_1', 'fastq_2', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'scDblFinder_score', 'scDblFinder_class', 'size_factors', 'leiden_res0_25', 'leiden_res1', 'leiden_res2', 'leiden_res2_25', 'leiden_res2_5', 'leiden_res3', 'size_factor2', 'manual_celltype_annotation', 'celltypist_cell_label', 'celltypist_conf_score', 'cellassign_predictions'
    var: 'gene_versions', 'gene_symbol', 'gene_name', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_deviant', 'binomial_deviance', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'na_gen

In [39]:
adata.obs

,sample,fastq_1,fastq_2,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_20_genes,total_counts_mt,log1p_total_counts_mt,...,leiden_res1,leiden_res2,leiden_res2_25,leiden_res2_5,leiden_res3,size_factor2,manual_celltype_annotation,celltypist_cell_label,celltypist_conf_score,cellassign_predictions
GCATGTACAATCTACG_Hair_graft_5_emptydrops_filter,Hair_graft_5_emptydrops_filter,nan,nan,8918,9.095939,53021.0,10.878462,20.184455,1558.0,7.351800,...,5,8,7,7,12,4.887130,IFE Spinous/Granular,Differentiated_KC,1.000000,sebaceous/apocrine
CATCGAAGTTCCAACA_Hair_graft_5_emptydrops_filter,Hair_graft_5_emptydrops_filter,nan,nan,7895,8.974112,47508.0,10.768674,15.963627,1647.0,7.407318,...,6,1,18,20,22,4.378977,IFE Mitotic,Differentiated_KC,0.999999,sebaceous/apocrine
CCAGCGATCCGAACGC_Hair_graft_5_emptydrops_filter,Hair_graft_5_emptydrops_filter,nan,nan,6499,8.779557,44857.0,10.711257,22.460263,1598.0,7.377134,...,4,4,3,16,18,4.134625,IFE Spinous/Granular,Differentiated_KC,1.000000,sebaceous/apocrine
GAAATGATCAATACCG_Hair_graft_5_emptydrops_filter,Hair_graft_5_emptydrops_filter,nan,nan,6928,8.843471,43752.0,10.686315,21.989852,2162.0,7.679251,...,4,4,3,14,15,4.032774,IFE Spinous/Granular,Differentiated_KC,0.999849,sebaceous/apocrine
GTACTTTCAACAACCT_Hair_graft_5_emptydrops_filter,Hair_graft_5_emptydrops_filter,nan,nan,5999,8.699515,41220.0,10.626703,29.553615,2967.0,7.995644,...,4,4,3,16,18,3.799390,IFE Spinous/Granular,Differentiated_KC,1.000000,sebaceous/apocrine
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GGCTCGAAGAGCTATA_Hair_graft_5_emptydrops_filter,Hair_graft_5_emptydrops_filter,nan,nan,301,5.710427,505.0,6.226537,23.762376,0.0,0.000000,...,1,10,9,8,5,0.046548,NaN,Differentiated_KC,0.998431,bulge
GGTATTGAGAGGTAGA_Hair_graft_5_emptydrops_filter,Hair_graft_5_emptydrops_filter,nan,nan,318,5.765191,482.0,6.180017,23.236515,0.0,0.000000,...,1,10,9,8,5,0.044428,NaN,Differentiated_KC,0.999982,IFE spinous 2
CTGAAACCAAGCTGGA_Hair_graft_5_emptydrops_filter,Hair_graft_5_emptydrops_filter,nan,nan,350,5.860786,485.0,6.186209,17.319588,15.0,2.772589,...,10,13,13,13,13,0.044704,Bulge/Isthmus,Undifferentiated_KC,0.946382,isthmus
CTACACCCATCACGAT_Hair_graft_5_emptydrops_filter,Hair_graft_5_emptydrops_filter,nan,nan,346,5.849325,481.0,6.177944,21.205821,10.0,2.397895,...,7,6,4,3,2,0.044335,Melanocytes,Melanocyte,0.986543,melanocytes


In [10]:
adata.obs.loc[:,'sample'].unique()

['Hair_graft_5_emptydrops_filter']
Categories (1, object): ['Hair_graft_5_emptydrops_filter']

In [18]:
adata.obs.loc[:,'sample'].count()

892

In [11]:
import rpy2
%load_ext rpy2.ipython

In [15]:
%%R -o sce

# Read the combined, empty-drop filtered count matrix RDS file and output SingleCellExperiment object to Python
rds_path <- "../../raw_data_processing/results/alevin/mtx_conversions/combined_emptydrops_filter_matrix.sce.rds"
sce <- readRDS(rds_path)

R[write to console]: Loading required package: SingleCellExperiment

R[write to console]: Loading required package: SummarizedExperiment

R[write to console]: Loading required package: MatrixGenerics

R[write to console]: Loading required package: matrixStats

R[write to console]: 
Attaching package: ‘MatrixGenerics’


R[write to console]: The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowC

In [16]:
# Convert the SingleCellExperiment object to AnnData object
import anndata2ri
from rpy2.robjects import r
from rpy2.robjects.conversion import localconverter

with localconverter(anndata2ri.converter):
    adata_raw = r('as(sce, "SingleCellExperiment")')

print(adata_raw)

AnnData object with n_obs × n_vars = 1189 × 236796
    obs: 'sample', 'fastq_1', 'fastq_2'
    var: 'gene_versions'


In [27]:
adata_raw.obs['sample']

GCATGTACAATCTACG_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
TGACAACAGATGTGGC_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
CATCGAAGTTCCAACA_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
AAATGCCAGCTGCCCA_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
CCAGCGATCCGAACGC_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
                                                                ...              
TTGAACGGTTCGCTAA_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
TGAGCATAGAAGATTC_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
CTACACCGTTAAGGGC_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
CGATTGATCTGTGCAA_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
ACCTTTAGTTTAGGAA_Hair_graft_5_emptydrops_filter    Hair_graft_5_emptydrops_filter
Name: sample, Length: 1189, dtype: category
Categories (1, object): ['Hair_graft_5_emptydrops_filt

In [17]:
adata_raw.obs['sample'].count()

1189

In [19]:
adata_raw.obs['sample'].unique()

['Hair_graft_5_emptydrops_filter']
Categories (1, object): ['Hair_graft_5_emptydrops_filter']

In [ ]:
%%R -o sce_s4_filtered

# Read the Hair graft 4, empty-drop filtered count matrix RDS file and output SingleCellExperiment object to Python
rds_path <- "../../raw_data_processing/results/alevin/mtx_conversions/Hair_graft_4/Hair_graft_4_emptydrops_filter_matrix.sce.rds"
sce_s4_filtered <- readRDS(rds_path)

In [21]:
# Convert the SingleCellExperiment object to AnnData object
with localconverter(anndata2ri.converter):
    adata_s4_filtered = r('as(sce_s4_filtered, "SingleCellExperiment")')

print(adata_s4_filtered)

AnnData object with n_obs × n_vars = 4117 × 236796
    obs: 'sample'
    var: 'gene_versions'


In [26]:
adata_s4_filtered.obs['sample']

AGATTGCTCCCTCAGT    Hair_graft_4
CTCGAAACATTCGACA    Hair_graft_4
TACTTACCACCATCCT    Hair_graft_4
ATCATCTAGTGTACTC    Hair_graft_4
GGCTCGACAGACACTT    Hair_graft_4
                        ...     
TACCTATGTGAAGGCT    Hair_graft_4
AGCTCCTCAATGAATG    Hair_graft_4
TGGCGCAAGAGTCTGG    Hair_graft_4
ACGGGCTTCATGCAAC    Hair_graft_4
CGAGCCATCAGTGTTG    Hair_graft_4
Name: sample, Length: 4117, dtype: category
Categories (1, object): ['Hair_graft_4']

In [22]:
%%R -o sce_s5_filtered

# Read the Hair graft 5, empty-drop filtered count matrix RDS file and output SingleCellExperiment object to Python
rds_path <- "../../raw_data_processing/results/alevin/mtx_conversions/Hair_graft_5/Hair_graft_5_emptydrops_filter_matrix.sce.rds"
sce_s5_filtered <- readRDS(rds_path)

In [23]:
# Convert the SingleCellExperiment object to AnnData object
with localconverter(anndata2ri.converter):
    adata_s5_filtered = r('as(sce_s5_filtered, "SingleCellExperiment")')

print(adata_s5_filtered)

AnnData object with n_obs × n_vars = 1189 × 236796
    obs: 'sample'
    var: 'gene_versions'


In [25]:
adata_s5_filtered.obs['sample']

GCATGTACAATCTACG    Hair_graft_5
TGACAACAGATGTGGC    Hair_graft_5
CATCGAAGTTCCAACA    Hair_graft_5
AAATGCCAGCTGCCCA    Hair_graft_5
CCAGCGATCCGAACGC    Hair_graft_5
                        ...     
TTGAACGGTTCGCTAA    Hair_graft_5
TGAGCATAGAAGATTC    Hair_graft_5
CTACACCGTTAAGGGC    Hair_graft_5
CGATTGATCTGTGCAA    Hair_graft_5
ACCTTTAGTTTAGGAA    Hair_graft_5
Name: sample, Length: 1189, dtype: category
Categories (1, object): ['Hair_graft_5']

In [32]:
adata_s4_filtered.obs_names = adata_s4_filtered.obs_names + '_' + adata_s4_filtered.obs['sample']
adata_s5_filtered.obs_names = adata_s5_filtered.obs_names + '_' + adata_s5_filtered.obs['sample']

In [34]:
adata_comb = anndata.concat([adata_s4_filtered, adata_s5_filtered])
adata_comb

AnnData object with n_obs × n_vars = 5306 × 236796
    obs: 'sample'

In [38]:
adata_comb.obs['sample'] = adata_comb.obs['sample'].astype('category')
adata_comb.obs['sample']

AGATTGCTCCCTCAGT_Hair_graft_4    Hair_graft_4
CTCGAAACATTCGACA_Hair_graft_4    Hair_graft_4
TACTTACCACCATCCT_Hair_graft_4    Hair_graft_4
ATCATCTAGTGTACTC_Hair_graft_4    Hair_graft_4
GGCTCGACAGACACTT_Hair_graft_4    Hair_graft_4
                                     ...     
TTGAACGGTTCGCTAA_Hair_graft_5    Hair_graft_5
TGAGCATAGAAGATTC_Hair_graft_5    Hair_graft_5
CTACACCGTTAAGGGC_Hair_graft_5    Hair_graft_5
CGATTGATCTGTGCAA_Hair_graft_5    Hair_graft_5
ACCTTTAGTTTAGGAA_Hair_graft_5    Hair_graft_5
Name: sample, Length: 5306, dtype: category
Categories (2, object): ['Hair_graft_4', 'Hair_graft_5']